# 🎯 Gradient Boosting - Version sans bruit (CORRIGÉE)

## ⚠️ Correction Data Leak

**Version précédente :** Nettoyage des données sur tout le dataset PUIS split train/val.
❌ **Problème** : Le set de validation était aussi "nettoyé" (suppression des cas difficiles/contradictoires), ce qui donnait un score faussement élevé ne reflétant pas la réalité.

**Version corrigée :**
1. Split Train / Validation d'abord
2. Nettoyage (suppression du bruit) UNIQUEMENT sur le **Train**
3. Évaluation sur le **Validation** original (qui contient toujours du bruit/incertitude)

---

## 📦 Import des bibliothèques

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    f1_score, 
    accuracy_score,
    roc_auc_score
)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

---

## 📊 1. Chargement et Split des données (CRUCIAL)

In [5]:
# Chargement
train_df_full = pd.read_csv('conversion_data_train.csv')
test_df = pd.read_csv('conversion_data_test.csv')

print(f"📊 Données totales : {train_df_full.shape[0]:,} lignes")

# ⚠️ SPLIT AVANT TOUT TRAITEMENT
# On réserve 20% des données pour la validation. Ces données NE DOIVENT PAS être nettoyées.
X_train_raw, X_val_raw, y_train_raw, y_val_raw = train_test_split(
    train_df_full.drop('converted', axis=1), 
    train_df_full['converted'], 
    test_size=0.2, 
    random_state=42, 
    stratify=train_df_full['converted']
)

print(f"\n1️⃣ Split Train/Val effectué :")
print(f"  Train (à nettoyer) : {len(X_train_raw):,} lignes")
print(f"  Val (intouché)     : {len(X_val_raw):,} lignes")

📊 Données totales : 284,580 lignes

1️⃣ Split Train/Val effectué :
  Train (à nettoyer) : 227,664 lignes
  Val (intouché)     : 56,916 lignes


---

## 🧹 2. Suppression du bruit (UNIQUEMENT sur le Train)

In [6]:
# Reconstitution temporaire pour le nettoyage
train_to_clean = X_train_raw.copy()
train_to_clean['converted'] = y_train_raw

# Identification des groupes sur le TRAIN seulement
feature_cols = ['country', 'age', 'new_user', 'source', 'total_pages_visited']
group_stats = train_to_clean.groupby(feature_cols)['converted'].agg(['count', 'mean']).reset_index()

# Groupes cohérents (mean = 0 ou 1)
coherent_groups = group_stats[(group_stats['mean'] == 0) | (group_stats['mean'] == 1)]

# Filtrage : On ne garde que les lignes qui appartiennent à des groupes cohérents
train_clean = train_to_clean.merge(coherent_groups[feature_cols], on=feature_cols, how='inner')

# Séparation X/y après nettoyage
X_train_clean = train_clean.drop(['converted', 'count', 'mean'], axis=1)
y_train_clean = train_clean['converted']

print(f"2️⃣ Nettoyage du Train set :")
print(f"  Lignes AVANT : {len(train_to_clean):,}")
print(f"  Lignes APRÈS : {len(train_clean):,}")
print(f"  🗑️ Supprimées : {len(train_to_clean) - len(train_clean):,} lignes (Bruit/Conflits)")

KeyError: "['count', 'mean'] not found in axis"

---

## 🔧 3. Feature Engineering (Appliqué partout)

In [ ]:
def feature_engineering(df):
    df_eng = df.copy()
    df_eng['interaction_age_pages'] = df_eng['age'] * df_eng['total_pages_visited']
    df_eng['is_active'] = (df_eng['total_pages_visited'] > 2).astype(int)
    df_eng['pages_per_age'] = df_eng['total_pages_visited'] / (df_eng['age'] + 0.1)
    return df_eng

# Application sur Train (Clean), Val (Raw) et Test
X_train_final = feature_engineering(X_train_clean)
X_val_final = feature_engineering(X_val_raw)
X_test_final = feature_engineering(test_df)

print("✅ Feature engineering appliqué sur tous les sets")

---

## 🎯 4. Entraînement et Évaluation

In [ ]:
numeric_features = ['age', 'total_pages_visited', 'interaction_age_pages', 'pages_per_age']
categorical_features = ['country', 'source']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough'
)

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=4,
        random_state=42
    ))
])

print("🎯 Entraînement sur TRAIN NETTOYÉ...")
model.fit(X_train_final, y_train_clean)
print("✅ Modèle entraîné")

In [ ]:
# Évaluation sur le VAL SET (Original, avec bruit)
print("📊 Évaluation sur VAL SET (Non nettoyé, représentatif du réel)...")
y_pred = model.predict(X_val_final)
y_pred_proba = model.predict_proba(X_val_final)[:, 1]

# Métriques
accuracy = accuracy_score(y_val_raw, y_pred)
f1 = f1_score(y_val_raw, y_pred)
roc_auc = roc_auc_score(y_val_raw, y_pred_proba)

print("\nRÉSULTATS :")
print("=" * 30)
print(f"  Accuracy  : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"  F1-Score  : {f1:.4f}")
print(f"  ROC-AUC   : {roc_auc:.4f}")

### Analyse des Résultats

Si le F1-Score a légèrement **baissé** par rapport à la version précédente (leakée), c'est **NORMAL**.
- Avant : On évaluait sur des données "faciles" (sans contradictions).
- Maintenant : On évalue sur des données réelles (avec contradictions).

Cependant, le modèle entraîné sur des données propres généralise souvent **mieux** sur les nouveaux cas car il n'a pas essayé d'apprendre le bruit contradictoire.

---

## 🎲 5. Prédictions finales

Pour la soumission finale, nous pouvons utiliser 2 stratégies :
1. Entraîner sur `Trainnettoyé` (pour éviter le bruit)
2. Ou entraîner sur `Train_Full` (plus de données, mais bruité)

Ici, nous restons cohérents avec notre approche "sans bruit".

In [ ]:
# Prédiction avec le modèle entraîné sur train_clean
test_predictions = model.predict(X_test_final)

# Sauvegarde
submission = pd.DataFrame({'converted': test_predictions})
submission.to_csv('predictions_sans_bruit_fixed.csv', index=False)
print("✅ 'predictions_sans_bruit_fixed.csv' généré")